# SkillPulse forecast model evaluation

This notebook reads the evaluated model summaries from a running SkillPulse API and then independently calls the backend's existing rolling-origin evaluator on a small selection of skill series. It does not introduce a second forecasting implementation.

Before running, start the API with a populated database (for example, run `python scripts/seed_database.py` from the repository root). The included job-posting dataset is synthetic, so the resulting validation measures are for pipeline verification and not real-world predictive claims.

In [ ]:
import json
import os
import sys
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "backend" / "app" / "ml" / "forecasting.py").is_file():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not locate backend/app/ml/forecasting.py; open this notebook inside the SkillPulse project.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

API_BASE = os.environ.get("SKILLPULSE_API", "http://localhost:8000/api").rstrip("/")

def get_json(path):
    with urlopen(f"{API_BASE}{path}", timeout=30) as response:
        return json.load(response)

print(f"Repository: {ROOT}")
print(f"API: {API_BASE}")

## Stored rolling-origin results

The endpoint summarizes all trained candidate models. MAE and RMSE are expressed in demand-share units; lower values indicate smaller forecast errors. MAPE is calculated only where actual demand share is at least 1%.

In [ ]:
performance = get_json("/analytics/model-performance")
if not performance.get("available"):
    raise RuntimeError(performance.get("message", "Forecast evaluation is unavailable. Train models by running the pipeline first."))

model_frame = pd.DataFrame(performance.get("models", []))
display(performance.get("validation", {}))
display(performance.get("selected_summary", {}))
display(model_frame.sort_values("mean_rmse").reset_index(drop=True))

if not model_frame.empty:
    model_frame.sort_values("mean_rmse").plot(
        x="model", y="mean_rmse", kind="bar", legend=False, figsize=(9, 4), color="#4f46e5",
        title="Mean rolling-origin RMSE by model",
    )
    plt.ylabel("Mean RMSE (demand-share units)")
    plt.xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Re-evaluate a small sample with the backend implementation

The following cells fetch up to eight forecastable skills and their quarterly demand series from the API, then run the same backend evaluator against each candidate. Skills with missing quarters or too little history are skipped rather than silently filling gaps.

In [ ]:
from app.ml.forecasting import MODEL_ORDER, evaluate_model

forecast_page = get_json("/forecasts?page=1&page_size=8&sort=change_2y&order=desc")
skill_ids = [row["skill_id"] for row in forecast_page.get("items", []) if row.get("status") == "ok"]
if not skill_ids:
    raise RuntimeError("No skills currently have forecasts. Run the full analysis pipeline first.")

query = urlencode({"skill_ids": ",".join(map(str, skill_ids)), "granularity": "quarter"})
series_response = get_json(f"/trends/series?{query}")
print(f"Requested {len(skill_ids)} skill series; returned {len(series_response.get('series', []))}.")

In [ ]:
rows = []
skipped = []
for item in series_response.get("series", []):
    raw = item.get("frequency", [])
    y = np.asarray([np.nan if value is None else float(value) for value in raw], dtype=float)
    if len(y) < 12 or not np.isfinite(y).all() or sum(item.get("mentions", [])) < 40:
        skipped.append(item["name"])
        continue
    for model_name in MODEL_ORDER:
        result = evaluate_model(model_name, y)
        rows.append({
            "skill": item["name"], "model": model_name, "folds": result.n_folds,
            "mae": result.mae, "rmse": result.rmse, "mape": result.mape, "r2": result.r2,
        })

rolling_scores = pd.DataFrame(rows)
if skipped:
    print("Skipped skills with insufficient history, sparse mentions or missing quarters:", ", ".join(skipped))
if rolling_scores.empty:
    print("No selected series met the backend evaluator's minimum data requirements.")
else:
    display(rolling_scores.sort_values(["skill", "rmse"]).reset_index(drop=True))
    display(rolling_scores.groupby("model")["rmse"].agg(["count", "mean", "median"]).sort_values("mean"))
    rolling_scores.groupby("model")["rmse"].mean().sort_values().plot(
        kind="bar", figsize=(9, 4), color="#0891b2", title="Sampled skills: backend rolling-origin mean RMSE",
    )
    plt.ylabel("Mean RMSE (demand-share units)")
    plt.xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Reading the results

- Cross-validation uses past quarters to predict subsequent quarters; folds are time-ordered and are never shuffled.
- A low error on this generated benchmark does not guarantee future performance on other datasets or labor markets.
- Compare each model with the naive baseline and inspect individual skill series before making decisions. The application reports uncertainty intervals and confidence separately from point forecasts.
- The sampled re-evaluation is intentionally limited to a few skills. The stored API summary is the broader evaluation result for the trained dataset.